# ReceiptGuard-ML Siamese Fraud Detection Training

This notebook trains the **Model 2 (Siamese Network)** for receipt fraud detection on Kaggle GPU.

## Goal
Detect fraudulent receipts by comparing pairs of receipts to find duplicates or tampered versions (date, total, or company changes).

## Setup

1. **Data**: Uses SROIE2019 dataset from Kaggle
2. **Model**: Siamese LayoutLM (shared weights)
3. **Hardware**: Kaggle GPU (T4/P100)

## Strategy
1. Preprocess raw SROIE data into tokens and bounding boxes.
2. Generate synthetic fraud pairs (50% legit, 50% fraud).
3. Fine-tune LayoutLM with a Siamese classification head.

In [ ]:
# Install dependencies
!pip install transformers torch torchvision pillow sentencepiece tiktoken
!pip install huggingface_hub accelerate scikit-learn matplotlib

In [ ]:
# Clone the repository
!git clone https://github.com/MoeenUddin01/Receipt_Guard.git
%cd Receipt_Guard

In [ ]:
# Setup environment
import sys
import os
from pathlib import Path
import gc
import torch

# Clear memory from previous runs
gc.collect()
torch.cuda.empty_cache()

# Add project root to path for imports
sys.path.append('/kaggle/working/Receipt_Guard')
os.environ['PYTHONPATH'] = '/kaggle/working/Receipt_Guard'

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")

## Step 1: Preprocessing
The Siamese model depends on processed SROIE samples. We run the Model 1 preprocessing pipeline first.

In [ ]:
# Run preprocessing pipeline
# Path to SROIE dataset on Kaggle (update this if your dataset path is different)
RAW_DATA_PATH = '/kaggle/input/datasets/urbikn/sroie-datasetv2/SROIE2019'
PROCESSED_PATH = '/kaggle/working/dataset/processed'

!mkdir -p {PROCESSED_PATH}

print("Running preprocessing...")
!python3 src/pipelines/preprocessing_pipeline.py --raw_path {RAW_DATA_PATH} --processed_path {PROCESSED_PATH}

## Step 2: Siamese Training
Now we run the Model 2 training pipeline which handles pair generation and Siamese fine-tuning.

In [ ]:
from src_2.pipelines.siamese_training_pipeline import run_siamese_training_pipeline, SiameseTrainingConfig

# Configure Siamese Training
config = SiameseTrainingConfig(
    model_path="microsoft/layoutlm-base-uncased",
    processed_path=PROCESSED_PATH,
    output_dir="/kaggle/working/artifacts/siamese",
    num_epochs=15,
    batch_size=4, # Reduced to 4 to ensure it fits on T4 GPU
    max_length=512,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    seed=42
)

print("Starting Siamese Training Pipeline...")
try:
    summary = run_siamese_training_pipeline(config)
    
    if summary.get('status') == 'completed':
        print("\n" + "=" * 60)
        print("SUCCESS: Siamese training completed!")
        print(f"Best threshold: {summary.get('best_threshold', 'N/A')}")
        print(f"Artifacts saved to: {config.output_dir}")
        print("=" * 60)
    else:
        print(f"\nERROR: Training failed: {summary.get('error')}")
except Exception as e:
    print(f"\nCRITICAL ERROR: {str(e)}")
    import traceback
    traceback.print_exc()

## Step 3: Distribution Analysis
After training, check the generated plots to see how well the model separates Legitimate and Fraudulent similarity distributions.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

plot_dir = Path("/kaggle/working/artifacts/siamese/plots")
plots = list(plot_dir.glob("*.png"))

if plots:
    # Show the last distribution plot
    latest_plot = sorted(plots)[-1]
    print(f"Displaying distribution: {latest_plot.name}")
    img = mpimg.imread(str(latest_plot))
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print("No plots found.")